# GLiNER large -- helio-pii multi-label eval

Loads `urchade/gliner_large-v2.1`, drives it with the `configs/taxonomy.json`
zero-shot prompts, applies each canonical label's own threshold from
`configs/thresholds.json`, and scores it against the
`data/test_harness.jsonl` multi-label classification harness.

In [ ]:
import json
import sys

import pandas as pd
from gliner import GLiNER

sys.path.append("../src")
from taxonomy import load_taxonomy
from metrics import evaluate

taxonomy = load_taxonomy("../configs/taxonomy.json", "../configs/thresholds.json")
taxonomy.thresholds

In [ ]:
import torch

MODEL_ID = "urchade/gliner_large-v2.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GLiNER.from_pretrained(MODEL_ID).to(device)
print(f"loaded {MODEL_ID} on {device}")

In [ ]:
with open("../data/test_harness.jsonl") as f:
    docs = [json.loads(line) for line in f]
len(docs)

In [ ]:
def predict_spans(model, text: str) -> list[dict]:
    """Raw span hits with each prompt's own canonical label attached, for
    eyeballing what actually fired. GLiNER's labels ARE the prompt strings,
    so multiple prompt_labels can fire on the same canonical category. Takes
    `model` explicitly (rather than closing over a global) so the same
    taxonomy/prompts can be run through more than one checkpoint for an
    A/B comparison further down."""
    raw = model.predict_entities(text, taxonomy.all_prompts, threshold=taxonomy.score_floor, multi_label=True)
    for e in raw:
        e["canonical"] = taxonomy.prompt_to_canonical[e["label"]]
    return raw


def predict_canonical_scores(model, text: str) -> dict[str, float]:
    """Doc-level score per canonical label, before any threshold is applied:
    the max prompt score seen anywhere in the text for that label. Kept
    separate from thresholding so the same scores can be reused to sweep
    thresholds afterwards, without re-running the model."""
    best: dict[str, float] = {}
    for e in predict_spans(model, text):
        c = e["canonical"]
        if e["score"] > best.get(c, -1.0):
            best[c] = e["score"]
    return best


def labels_from_scores(scores: dict[str, float]) -> list[str]:
    """A canonical label with no span clearing its threshold is simply
    absent -- that's the "non-PII" case, no extra work needed."""
    return [c for c, score in scores.items() if score >= taxonomy.threshold_for(c)]

In [ ]:
import time

from tqdm.auto import tqdm


def score_harness(model, docs: list[dict]) -> tuple[list[dict], list[list[str]]]:
    """Run one checkpoint over the whole harness once, returning the raw
    per-doc canonical scores (for threshold sweeping) and the thresholded
    predictions (for evaluate())."""
    canonical_scores, predictions = [], []
    start = time.monotonic()
    for d in tqdm(docs, desc="scoring documents", unit="doc"):
        scores = predict_canonical_scores(model, d["text"])
        canonical_scores.append(scores)
        predictions.append(labels_from_scores(scores))
    elapsed = time.monotonic() - start
    print(f"scored {len(docs)} docs in {elapsed:.1f}s ({elapsed / len(docs):.2f}s/doc)")
    return canonical_scores, predictions


canonical_scores, predictions = score_harness(model, docs)
for d, pred in zip(docs, predictions):
    print(f"{d['id']:24s} gold={d['labels']!s:30s} pred={pred}")

In [ ]:
gold_label_sets = [d["labels"] for d in docs]
rows, micro, macro = evaluate(gold_label_sets, predictions, taxonomy.canonical_names)

per_label_df = pd.DataFrame(rows).sort_values("label").reset_index(drop=True)
per_label_df

In [ ]:
print("micro:", micro)  # dominated by frequent labels
print("macro:", macro)  # every label weighted equally -- watch this for rare PHI labels

In [ ]:
# The harness tags each row with scenario_type/noise_type/polarity -- break
# exact-match accuracy down by scenario_type to see where it's weakest
# (e.g. boundary_ambiguous vs. near_miss_negative).
harness_df = pd.DataFrame(docs)
harness_df["predicted"] = predictions
harness_df["exact_match"] = [set(g) == set(p) for g, p in zip(gold_label_sets, predictions)]
harness_df.groupby("scenario_type")["exact_match"].mean()

## Threshold tuning

`configs/thresholds.json` is tagged `"initial-operating-profile"` -- it was
never tuned. The sweep below reuses the scores already computed above (no
extra model calls) to find, per canonical label, the threshold with the
best F1.

**Caveat:** this tunes on the same harness you're evaluating on, which
overfits to it. If the harness is large enough, split it into a tuning set
and a held-out set first and only trust the eval metrics from the held-out
half.

In [ ]:
from metrics import sweep_thresholds

tuned_rows = sweep_thresholds(gold_label_sets, canonical_scores, taxonomy.canonical_names)
tuned_df = pd.DataFrame(tuned_rows).sort_values("label").reset_index(drop=True)
tuned_df

In [ ]:
tuned_config = {
    "schema_version": 1,
    "status": "tuned-on-test-harness",
    "model": "urchade/gliner_large-v2.1",
    "taxonomy": "configs/taxonomy.json",
    "default_threshold": taxonomy.default_threshold,
    "thresholds": {row["label"]: row["threshold"] for row in tuned_rows},
}
with open("../configs/thresholds.tuned.json", "w") as f:
    json.dump(tuned_config, f, indent=2)
print("wrote ../configs/thresholds.tuned.json -- review, then replace configs/thresholds.json if it looks right")

## Error inspection

The worst-F1 labels from the (untuned) run above, with a few concrete false
positive / false negative examples each -- this is usually more actionable
than the aggregate numbers for deciding whether a `prompt_labels` entry is
too broad (FPs) or too narrow (FNs).

In [ ]:
def label_errors(label: str):
    fps, fns = [], []
    for d, pred in zip(docs, predictions):
        gold, predset = set(d["labels"]), set(pred)
        if label in predset and label not in gold:
            fps.append(d)
        elif label in gold and label not in predset:
            fns.append(d)
    return fps, fns


worst_labels = per_label_df.sort_values("f1").head(3)["label"].tolist()
for label in worst_labels:
    fps, fns = label_errors(label)
    print(f"=== {label}: {len(fps)} false positives, {len(fns)} false negatives ===")
    for d in fps[:3]:
        print(f"  FP [{d['scenario_type']}] {d['id']}: {d['text'][:100]!r}")
    for d in fns[:3]:
        print(f"  FN [{d['scenario_type']}] {d['id']}: {d['text'][:100]!r}")

## A/B: swap the backbone checkpoint

`gliner_multi_pii-v1` is fine-tuned specifically on PII, but it's still a
GLiNER checkpoint: labels are natural-language prompts passed in at
inference time, not a fixed output head, so `taxonomy.all_prompts`
-- your own zero-shot prompts (e.g. "medical record number") included --
works against it exactly the same way, no taxonomy/threshold changes
needed for a same-inputs comparison.

This loads a second model and reruns the harness through `score_harness`,
so it costs a second full pass (same runtime as the first run, reported
above). If you're on a memory-constrained GPU, free the first model first
(`del model; torch.cuda.empty_cache()`) before loading the second.

In [ ]:
MODEL_B_ID = "urchade/gliner_multi_pii-v1"
model_b = GLiNER.from_pretrained(MODEL_B_ID).to(device)
print(f"loaded {MODEL_B_ID} on {device}")

canonical_scores_b, predictions_b = score_harness(model_b, docs)
rows_b, micro_b, macro_b = evaluate(gold_label_sets, predictions_b, taxonomy.canonical_names)

In [ ]:
# Same thresholds, same taxonomy, same harness for both -- the only
# variable is the backbone. Once you know which one wins, re-run the
# threshold sweep (above) for that model specifically before shipping it.
print(f"{MODEL_ID:35s} micro={micro}")
print(f"{MODEL_B_ID:35s} micro={micro_b}")
print(f"{MODEL_ID:35s} macro={macro}")
print(f"{MODEL_B_ID:35s} macro={macro_b}")

compare_df = pd.DataFrame(rows).set_index("label")[["precision", "recall", "f1"]].add_suffix("_large_v2_1")
compare_df_b = pd.DataFrame(rows_b).set_index("label")[["precision", "recall", "f1"]].add_suffix("_multi_pii_v1")
compare_df.join(compare_df_b).sort_index()